**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/petersen-2020"], check=True)
    os.chdir("/content/encore/entries/petersen-2020")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings of petersen-2020-01, checked on DANDI:000059 version 0.261115.0311

Paper: Petersen PC, Buzsaki G (2020). Cooling of medial septum reveals theta phase lag coordination of hippocampal cell assemblies. Neuron 107, 731-744. doi:10.1016/j.neuron.2020.05.023.

The attempt petersen-2020-01 was made on version 0.250624.0444 and recorded 13 findings. The repeat attempt petersen-2020-02, made blind on version 0.261115.0311 (notebook `reproduce_petersen_2020.ipynb`), reports some of them again. This notebook checks each earlier finding directly on version 0.261115.0311, one short section per finding, each opening with the earlier statement. It reads only file metadata and arrays, except for finding 8, which loads the units and position of four sessions. Data are streamed from the archive with the entry's helper `ms_cooling.py`; nothing is written to disk.

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time, resource, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from scipy import stats
import ms_cooling as mc
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
T_START = time.time()

In [ ]:
sessions = mc.list_sessions()


def _text(v):
    return v.decode() if isinstance(v, bytes) else v


def survey(row):
    """File metadata of one session: trial states, unit quality and cell type,
    electrode locations, tracking rate, temperature series and descriptions."""
    f = mc.open_asset(row.asset_id)
    out = dict(rat=row.rat, session=row.session, has_trials="intervals/trials" in f)
    if out["has_trials"]:
        t = f["intervals/trials"]
        out["n_trials"] = len(t["start_time"])
        out["states"] = pd.Series(mc._dec(t["cooling_state"][:])).value_counts().to_dict()
        out["state_desc"] = _text(t["cooling_state"].attrs.get("description", ""))
    out["has_units"] = "units" in f
    if out["has_units"]:
        u = f["units"]
        out["quality"] = pd.Series(mc._dec(u["quality"][:])).value_counts().to_dict()
        q = mc._dec(u["quality"][:])
        ct = mc._dec(u["cell_type"][:])
        out["good_typed"] = int(((q == "good") & ~np.isin(ct, ["", "Unknown"])).sum())
    out["location"] = pd.Series(mc._dec(f["general/extracellular_ephys/electrodes/location"][:])).value_counts().to_dict()
    b = f["processing/behavior"]
    ts = b["SubjectPosition/SpatialSeries/timestamps"][:]
    out["pos_hz"] = round(float(1 / np.median(np.diff(ts))), 1)
    out["pos_desc"] = _text(b["SubjectPosition"].attrs.get("description", ""))
    out["temp"] = "measured" if "MeasuredTemperature" in b else "modelled"
    g = b["MeasuredTemperature" if out["temp"] == "measured" else "ModelledTemperature"]
    out["temp_desc"] = _text(g.attrs.get("description", ""))
    out["cooling_desc"] = _text(f["intervals/cooling"].attrs.get("description", "")) if "intervals/cooling" in f else ""
    f.close()
    return out


with ThreadPoolExecutor(6) as ex:
    INFO = pd.DataFrame(list(ex.map(survey, sessions.itertuples())))

In [ ]:
print(f"dandiset {mc.DANDISET} version {mc.VERSION}: {len(sessions)} NWB files, one per session")
print("sessions per rat:", INFO.groupby("rat").size().to_dict())
print("files with a trials table:", INFO.has_trials.sum(), "; with units:", INFO.has_units.sum())
print("temperature series:", INFO.temp.value_counts().to_dict())
print("tracking rate (Hz) by rat:", {r: g.pos_hz.value_counts().to_dict() for r, g in INFO.groupby("rat")})
print("trial states over all files:", pd.DataFrame(list(INFO.states)).sum().astype(int).to_dict())
print("electrode locations by file:", pd.Series([str(d) for d in INFO.location]).value_counts().to_dict())
print(f"survey in {time.time() - T_START:.0f} s")

The checks below use this survey, the dandiset's release notes for this version, and, for finding 8, the units and position of four sessions.

In [ ]:
META = mc._get_json(f"{mc.API}/dandisets/{mc.DANDISET}/versions/{mc.VERSION}/info/")["metadata"]
NOTES = META["releaseNotes"]
print("release notes,", len(NOTES), "characters; lines naming sessions not in the dandiset or not converted:")
for line in NOTES.splitlines():
    if line.startswith("- Not"):
        print(" ", line)

## Earlier finding 1: temperature of MS10 and the 2018-07-19 sessions

> The temperature series of the nine MS10 sessions and of the two control sessions of 2018-07-19 (MS21 and MS22) is not a recording. The paper's code lists no temperature channel for these sessions. In the files ten of them run from 17.0 to 37.0 C (exactly in nine; MS10-170313 reaches 16.98 C) and sit at exactly 37.0 C for 16% to 35% of samples, and MS10-170307 is a constant 37.0 C, while no recorded trace repeats its maximum. Nothing in the files says the series is modelled.

In [ ]:
sel = INFO.session.str.startswith("MS10") | INFO.session.str.contains("180719")
print(INFO.loc[sel, ["session", "temp"]].to_string(index=False))
print()
print("description of the modelled series (start):", INFO.loc[INFO.temp == "modelled", "temp_desc"].iloc[0][:160])
print("files with a modelled series:", (INFO.temp == "modelled").sum())

**Result:** the MS10 series are now stored as `ModelledTemperature`, described as "NOT A MEASUREMENT", and the two 2018-07-19 sessions hold a measured series. MS10 still has no measured temperature, which petersen-2020-02 reports as its finding 3.

## Earlier finding 2: the 2018-07-19 sessions labelled as cooled

> The two sessions of 2018-07-19, which the paper's code lists as controls without cooling (cooling_session 0, no onsets), have trials labelled Pre-Cooling, Cooling on, Cooling off and Post-Cooling and a temperature dip; their theta frequency during 'Cooling on' differs from pre by -0.20 and +0.44 Hz, against a median of -0.82 Hz in the cooled sessions.

In [ ]:
for _, r in INFO[INFO.session.str.contains("180719")].iterrows():
    print(r.session, r.states)
    print("  cooling table description (end):", r.cooling_desc[-230:])

**Result:** both sessions still have Pre, Cooling and Post trials. Their descriptions now say that the measured temperature stays flat through the labelled cooling, and that the lab's labels are kept. petersen-2020-02 reports the labels as its finding 2.

## Earlier finding 3: sessions of the paper missing from the dandiset

> Six of the paper's 53 behavioural sessions are not in the dandiset (MS12-170712, MS12-170715, MS13-171204, MS13-171205, MS13-171206, MS21-180806); four of them were in version 0.210812.1514 as files without behaviour.

In [ ]:
EARLIER_MISSING = ["MS12-170712", "MS12-170715", "MS13-171204", "MS13-171205", "MS13-171206", "MS21-180806"]
present = {k: bool(INFO.session.str.startswith(k + "-").any()) for k in EARLIER_MISSING}
print("in this version:", present)
print("still missing:", sum(not v for v in present.values()), "of", len(present))

**Result:** MS12-170715 has been added; the other five sessions are still not in the dandiset. The release notes name four of them (MS12-170712, MS13-171204, MS13-171205 and MS13-171206) as not in the dandiset and do not mention MS21-180806.

## Earlier finding 4: MS21-180627 and MS21-180712

> MS21-180627 has trials without the cooling state and MS21-180712 has no trials, so two more of the paper's sessions cannot be used for the cooling comparisons.

In [ ]:
for k in ["MS21-180627", "MS21-180712"]:
    r = INFO[INFO.session.str.startswith(k)].iloc[0]
    print(r.session, "trials:", r.n_trials, r.states)

**Result:** both sessions now have trials with the Pre, Cooling and Post states: MS21-180627 has 179 trials (one 'not labelled') and MS21-180712 has 127.

## Earlier finding 5: cell types, place fields and electrode locations

> The files hold no cell types, no place field definitions and no electrode locations (every location is 'unknown'), so the putative pyramidal cells and interneurons and the place fields of the paper have to be made again.

In [ ]:
u = INFO[INFO.has_units]
print("files with units:", len(u), "; with at least one good unit with a cell type:", (u.good_typed > 0).sum())
loc_unknown = INFO[INFO.location.apply(lambda d: "unknown" in d)]
print("files whose electrodes are all 'unknown':", len(loc_unknown), loc_unknown.session.tolist())
print("files with other locations:", pd.Series([str(d) for d in INFO.location if "unknown" not in d]).value_counts().to_dict())

**Result:** the files now give cell types (in 23 of the 30 files with units, some good unit has a type) and electrode locations in 37 files. Fourteen files, all of MS21 or MS22, still give every electrode as 'unknown', and no file holds place fields; petersen-2020-02 reports the missing place fields and the untyped units as its finding 4.

## Earlier finding 6: the trial state 'Cooling off'

> The trial state 'Cooling off' (a median 14 trials per session between cooling and recovery) is not described in the files or the paper.

In [ ]:
states = pd.DataFrame(list(INFO.states)).sum().astype(int).to_dict()
print("trial states over all files:", states)
print("'Cooling off' present:", "Cooling off" in states)
print("description of cooling_state:", INFO.state_desc.iloc[0])

**Result:** no trial has the state 'Cooling off'. The trials between cooling and recovery are 'transition (excluded)', and the column's description says which windows these are.

## Earlier finding 7: sessions in which every unit is 'unsorted'

> In seven sessions (the six MS10 sessions with units and MS12-170723) every unit has quality 'unsorted', so no curated unit can be selected; MS10 has no usable units.

In [ ]:
u = INFO[INFO.has_units].copy()
u["good"] = u.quality.apply(lambda d: d.get("good", 0))
u["all_unsorted"] = u.quality.apply(lambda d: set(d) == {"unsorted"})
print(u.loc[u.session.str.startswith("MS10") | (u.session == "MS12-170723-141121"), ["session", "quality"]].to_string(index=False))
print("files with units, all 'unsorted':", u.session[u.all_unsorted].tolist())

**Result:** six of the seven MS10 sessions with units now have 'good' units (2 to 16 each). MS10-170321 and MS12-170723 still hold only 'unsorted' units; petersen-2020-02 counts them among the 23 sessions without curated units in its finding 4.

## Earlier finding 8: spatial information of the units of the 2018-06-21 sessions

> In MS21-180621 and MS22-180621 the units carry no spatial information (0.23 and 0.12 bits/spike, the same as with spikes shifted by 60 s), although they are locked to theta, so the clock of spikes and position could not be established there.

Spatial information (Skaggs, bits per spike) of the rate map of each good unit on the linearized position, in the trials, at running speed above 10 cm/s, with 5 cm bins smoothed as in the helper, against the same with the spikes shifted by 60 s (circularly within the trials' span). MS21-180625 and MS22-180629 are reference sessions of the same rats.

In [ ]:
def info_true_shifted(S, shift=60.0):
    b, tr, t = S["beh"], S["trials"], S["beh"]["t"]
    edges = np.arange(0, 350 + 5, 5.0)
    ti = np.searchsorted(tr.start.values, t, side="right") - 1
    ok = (ti >= 0) & (t <= tr.stop.values[np.clip(ti, 0, None)]) & np.isfinite(b["lin"])
    ok &= np.nan_to_num(b["speed"]) > mc.RUN_SPEED
    occ, _ = np.histogram(b["lin"][ok], bins=edges, weights=np.gradient(t)[ok])
    a, z = S["span"]
    res = []
    for U in S["units"]:
        row = []
        for sh in (0.0, shift):
            s = np.sort(np.mod(U["spikes"] - a + sh, z - a) + a)
            j = np.clip(np.searchsorted(t, s), 0, len(t) - 1)
            j = j[ok[j]]
            cnt, _ = np.histogram(b["lin"][j], bins=edges)
            raw = np.where(occ > 0, cnt / np.where(occ > 0, occ, 1), np.nan)
            row.append(mc.skaggs_info(mc.smooth_map(raw, occ), occ))
        res.append(row)
    return np.array(res)


rows = []
for name in ["MS21-180621-141817", "MS22-180621-131028", "MS21-180625-153927", "MS22-180629-110319"]:
    S = mc.load_session(sessions[sessions.session == name].iloc[0], with_lfp=False)
    I = info_true_shifted(S)
    rows.append(dict(session=name, good_units=len(I), info_median=np.nanmedian(I[:, 0]),
                     info_shift60_median=np.nanmedian(I[:, 1]),
                     units_info_above_shift=int((I[:, 0] > I[:, 1]).sum())))
    del S
F8 = pd.DataFrame(rows)
print(F8.round(3).to_string(index=False))

**Result:** in MS21-180621 and MS22-180621 the units still carry no more spatial information than with the spikes shifted by 60 s (medians in the table above), while in the two reference sessions the information falls more than tenfold with the shift.

## Earlier finding 9: sessions outside the paper's analysis

> The dandiset holds three sessions that are not in the paper's behavioural analysis (the two 2018-07-19 controls and MS10-170307, whose temperature is constant) without saying so; they carry the same cooling labels as the analysed sessions.

In [ ]:
desc = META["description"]
print("dandiset description mentions 'control', 'not analysed' or 'excluded':",
      any(w in desc.lower() for w in ["control", "not analy", "exclud"]))
for k in ["MS21-180719", "MS22-180719", "MS10-170307"]:
    r = INFO[INFO.session.str.startswith(k)].iloc[0]
    flat = "does not appear to have been cooled" in r.cooling_desc
    print(r.session, "| cooling table says the septum does not appear cooled:", flat,
          "| temperature:", r.temp, "| states:", r.states)

**Result:** the cooling tables of the two 2018-07-19 sessions now say that the septum does not appear to have been cooled. No description says that these sessions, or MS10-170307, are outside the paper's behavioural analysis; MS10-170307 has cooling states like the analysed sessions.

## Earlier finding 10: the trials table of the first published version

> The first published version (0.210812.1514) has no trials table in its files, so the cooling states and choice errors are only in the later versions.

This finding is about version 0.210812.1514, a published version that cannot change; it is not a property of version 0.261115.0311. What can be said of this version is printed.

In [ ]:
print("files of this version with a trials table:", INFO.has_trials.sum(), "of", len(INFO))

**Result:** not checked. Every file of version 0.261115.0311 has a trials table; the finding concerns the content of version 0.210812.1514, which this version does not change.

## Earlier finding 11: tracking rate of the MS10 sessions

> The Methods give position tracking at 120 Hz; the MS10 sessions are sampled at 40 Hz (six), 30 Hz (one) and 120 Hz (two).

In [ ]:
m10 = INFO[INFO.rat == "MS10"]
print(m10[["session", "pos_hz"]].to_string(index=False))
print("MS10 rates:", m10.pos_hz.value_counts().to_dict(), "; other rats:", INFO[INFO.rat != "MS10"].pos_hz.value_counts().to_dict())
print("release notes on MS10 tracking:", [l for l in NOTES.splitlines() if "OptiTrack" in l])

**Result:** six MS10 sessions are tracked at 40 Hz, one at 30 Hz and two at about 120 Hz (119.8), as before; the Methods give 120 Hz. The release notes say MS10's 120 Hz OptiTrack Take files were not converted.

## Earlier finding 12: the P value of Figure 5H

> The legend of Figure 5H gives P = 8.0e-5 by a sign rank test with N = 10 sessions; an exact signed-rank test over 10 sessions cannot give a P below 0.002, so the test must be over pairs or the N refers to something else.

The legend of Figure 5H reads "Pre vs cooling P = 8.0 −5; Post vs cooling P = 0.00032; N = 10 sessions, 4 rats", and the Results "P < 10 −4; sign rank test; n = 10 sessions". The paper is the same as in petersen-2020-01. The smallest two-sided P an exact signed-rank test over 10 paired values can give is computed here.

In [ ]:
x = np.arange(1, 11, dtype=float)
p_min = stats.wilcoxon(x, np.zeros(10), method="exact").pvalue
print(f"smallest exact two-sided signed-rank P with N = 10: {p_min:.5f} (2 / 2**10 = {2 / 2**10:.5f}); legend gives 8.0e-5")

**Result:** an exact signed-rank test over 10 sessions cannot give a P below 0.00195, so the legend's P = 8.0e-5 with N = 10 sessions is still inconsistent.

## Earlier finding 13: the 12% decrease of theta frequency

> The Discussion gives a 12% decrease of LFP theta frequency during cooling, while the Results give within-field LFP theta of 8.4 Hz before and 7.7 Hz during cooling, a decrease of about 8%, which this attempt reproduces (9% over trials).

The Discussion reads "During MS cooling, LFP theta frequency decreased by 12 %", and the Results give within-field LFP theta "f LFP pre = 8.4 ± 0.38, f LFP cooling = 7.7 ± 0.36" (n = 365). The paper is the same as in petersen-2020-01.

In [ ]:
print(f"decrease from 8.4 to 7.7 Hz: {100 * (8.4 - 7.7) / 8.4:.1f}%")

**Result:** the Results' values give a decrease of 8.3%, against the Discussion's 12%, as before. petersen-2020-02 tests the 12% as claim disc-theta-frequency-12-percent and finds 6.6 to 9.9% for the mean per-session decrease and 11.1% for the largest decrease of the aligned mean.

In [ ]:
print(f"notebook run time {(time.time() - T_START) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Summary

| earlier finding | on version 0.261115.0311 |
|---|---|
| 1 temperature of MS10 and 2018-07-19 not a recording | partly fixed: labelled as a model; MS10 has no measurement (petersen-2020-02 finding 3) |
| 2 2018-07-19 sessions labelled as cooled | partly fixed: labels kept, descriptions now say the temperature is flat (petersen-2020-02 finding 2) |
| 3 six of the paper's sessions missing | partly fixed: MS12-170715 added, five still missing |
| 4 MS21-180627 and MS21-180712 without states | fixed |
| 5 no cell types, place fields or locations | partly fixed: cell types and most locations added; 14 files 'unknown', no place fields (petersen-2020-02 finding 4) |
| 6 'Cooling off' undescribed | fixed |
| 7 sessions with only 'unsorted' units | partly fixed: two of seven remain (petersen-2020-02 finding 4) |
| 8 no spatial information in the 2018-06-21 sessions | still present |
| 9 sessions outside the paper's analysis not said | partly fixed: 2018-07-19 described as not cooled; nothing says which sessions the paper analysed |
| 10 no trials table in version 0.210812.1514 | not checked: about an earlier version |
| 11 MS10 tracked below 120 Hz | still present |
| 12 Figure 5H P value | still present |
| 13 12% against 8.4 to 7.7 Hz | still present |